# Ingest Sprints.json file
1. Read the file using Spark dataframe reader API
1. Define and enforce schema
1. Add Metadata Columns
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.Bronze-helpers


In [0]:

# Define source_file and table_name
source_file = f"{landing_folder_path}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

### Step 1 - Read the json file using the dataframe reader API

In [0]:
# Define the schema
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, StringType, DateType

results_schema = StructType([
    StructField('date', DateType()),
    StructField('raceName', StringType(),),
    StructField('round', IntegerType()),
    StructField('season', IntegerType(),),
    StructField('url', StringType()),
    StructField('constructorId', StringType(),),
    StructField('driverId', StringType()),
    StructField('grid', IntegerType(),),
    StructField('laps', IntegerType()),
    StructField('number', IntegerType(),),
    StructField('points', FloatType()),
    StructField('position', IntegerType(),),
    StructField('positionText', StringType()),
    StructField('status', StringType(),)
])
    

In [0]:
# Read data from the sprints file

sprints_df = (
    spark.read
        .format('json')
        .schema(results_schema)
        .option('mode','FAILFAST')
        .option('multiLine', True)
        .load(source_file)
)



In [0]:
display(sprints_df)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)


#### Step 3 - Write to bronze delta table

In [0]:
(
    sprints_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)     
)

In [0]:
display(spark.table(table_name))

In [0]:
%sql
select season, count(*)
from formula1.bronze.sprints
where season is not null
group by season
order by season